In [2]:
import time
from pymongo import MongoClient

# --- 1. CONFIGURACIÓN ---
NOMBRE_GRUPO = "G2_RealEstate_ConstanzaTorres" 

# IMPORTANTE: Cambia esto a "La Serena" cuando uses la URL de La Serena
CIUDAD_ACTUAL = "La Serena" 

try:
    client = MongoClient("database", 27017)
    db = client['ArriendosPrueba_IVRegión'] 
    # Usaremos una colección nueva para que veas los datos limpios y separados
    coleccion = db['G2_Arriendos']
    print(f"1. CONEXIÓN EXITOSA: Conectado para procesar {CIUDAD_ACTUAL}.")
except Exception as e:
    print(f"ERROR DE CONEXIÓN: {e}")

1. CONEXIÓN EXITOSA: Conectado para procesar La Serena.


In [3]:
import time
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("🚀 Iniciando prueba de extracción Avanzada (con Scroll y Filtros)...")

# 1. Configuración Headless (Para entornos de servidor/Docker/Jupyter)
options = Options()
options.add_argument("--headless=new") # Modo invisible obligatorio para tu entorno
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--window-size=1920,1080")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")

driver = webdriver.Chrome(options=options)

try:
    # 2. Navegación
    url = "https://www.portalinmobiliario.com/arriendo/departamento/la-serena-coquimbo"
    print(f"🌐 Navegando a: {url}")
    driver.get(url)

    # 3. Esperar resultados iniciales
    print("⏳ Esperando carga inicial...")
    WebDriverWait(driver, 15).until(
        EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
    )
    
    # --- LA MAGIA DEL SCROLL ---
    print("📜 Deslizando hasta el final de la página para cargar todo...")
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(5) # Le damos 5 segundos para que carguen las imágenes y textos rezagados

    # 4. Capturar bloques
    bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
    print(f"✅ Se detectaron {len(bloques)} bloques en total (incluyendo posibles anuncios).\n")
    print("-" * 50)

    # 5. Iterar sobre los primeros 10 bloques para ver si el filtro funciona
    propiedades_validas = 0
    
    for bloque in bloques[:10]:
        # Si ya encontramos 5 propiedades válidas, paramos la prueba
        if propiedades_validas >= 5:
            break
            
        try:
            # Extraemos los textos primero
            nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").text
            precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").text
            
            # --- FILTRO ANTI-BLANCOS ---
            # Si el nombre o el precio están vacíos, es publicidad o cargó mal. Lo ignoramos.
            if not nombre.strip() or not precio_texto.strip():
                print("   ⚠️ [Bloque ignorado: Publicidad o datos vacíos]")
                continue
                
            # Si pasamos el filtro, es una propiedad real
            propiedades_validas += 1
            print(f"🏠 PROPIEDAD VÁLIDA #{propiedades_validas}:")
            print(f"   ➤ Título: {nombre}")
            print(f"   ➤ Precio bruto: {precio_texto}")
            
            # Prueba de limpieza
            precio_limpio = precio_texto.replace("$", "").replace(".", "").strip()
            if precio_limpio.isdigit():
                print(f"   ➤ Precio limpio (para BD): {float(precio_limpio)}")
            else:
                print(f"   ➤ Precio limpio (para BD): 0.0 (No numérico)")
                
            print("-" * 50)
            
        except Exception:
            # Si ni siquiera tiene la clase del título, definitivamente es publicidad
            print("   ⚠️ [Bloque ignorado: Estructura no coincide]")
            continue

except Exception as e:
    print(f"🚨 Error crítico durante la prueba: {e}")

finally:
    # 6. Cierre seguro
    print("🧹 Prueba finalizada. Cerrando el navegador.")
    driver.quit()

🚀 Iniciando prueba de extracción Avanzada (con Scroll y Filtros)...
🌐 Navegando a: https://www.portalinmobiliario.com/arriendo/departamento/la-serena-coquimbo
⏳ Esperando carga inicial...
📜 Deslizando hasta el final de la página para cargar todo...
✅ Se detectaron 48 bloques en total (incluyendo posibles anuncios).

--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #1:
   ➤ Título: Arriendo Departamento Amoblado A Pasos Del Centro, La Serena
   ➤ Precio bruto: $
550.000
   ➤ Precio limpio (para BD): 550000.0
--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #2:
   ➤ Título: Departamento Avenida Del Mar Id: 169064
   ➤ Precio bruto: $
700.000
   ➤ Precio limpio (para BD): 700000.0
--------------------------------------------------
🏠 PROPIEDAD VÁLIDA #3:
   ➤ Título: Departamento Emilio Apey Id: 161974
   ➤ Precio bruto: $
500.000
   ➤ Precio limpio (para BD): 500000.0
--------------------------------------------------
   ⚠️ [Bloque ignorado: Publi

In [4]:
# --- PASO 0: LIMPIEZA TOTAL Y REPARACIÓN ---
import os
import time
from pymongo import MongoClient
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# 1. FORZAR LA PANTALLA PARA noVNC
# Esto le dice a Chrome en qué monitor virtual debe dibujarse para que VNC lo pueda grabar
os.environ["DISPLAY"] = ":99"  # IMPORTANTE: Si sigue negra, cambia ":0" por ":1" o ":99"

# Cierra procesos viejos que hayan quedado abiertos
os.system("pkill -9 chrome")
os.system("pkill -9 chromedriver")
os.system("rm -rf /tmp/.com.google.Chrome.*")
os.system("rm -rf /tmp/.org.chromium.Chromium.*")
print("Limpieza completada. Pantalla virtual configurada.")

# --- VARIABLES GENERALES ---
NOMBRE_GRUPO = "G2"
datos_finales = []   
driver = None        

# --- PASO 1: CONFIGURACIÓN DEL NAVEGADOR ---
options = Options()

# 🚨 ¡ATENCIÓN! Comentamos el "--headless=new" para que puedas verlo en noVNC 🚨
# options.add_argument("--headless=new")

# Argumentos de estabilidad obligatorios para entornos Docker/Servidor
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")
options.add_argument("--disable-software-rasterizer")
options.add_argument("--window-size=1920,1080")
options.add_argument(
    "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/120.0.0.0 Safari/537.36"
)

try:
    # Inicia el navegador
    driver = webdriver.Chrome(options=options)
    print("Navegador iniciado correctamente.")

    # --- PASO 2: NAVEGACIÓN Y EXTRACCIÓN ---
    limite_paginas = 3
    driver.get("https://www.portalinmobiliario.com/arriendo/departamento/la-serena-coquimbo")

    for nivel_pagina in range(limite_paginas):
        print(f"\n--- Procesando Página {nivel_pagina + 1} ---")

        # Espera explícita a que existan resultados
        WebDriverWait(driver, 20).until(
            EC.presence_of_all_elements_located((By.CSS_SELECTOR, ".ui-search-layout__item"))
        )
        
        # --- SCROLL HUMANO (para que pueda tomar todos los datos ---
        for _ in range(5):
            driver.execute_script("window.scrollBy(0, 800);")
            time.sleep(1)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(3) # Pausa final para que asienten los últimos bloques

        # Capturamos todos los bloques presentes en la página
        bloques = driver.find_elements(By.CSS_SELECTOR, ".ui-search-layout__item")
        
        propiedades_extraidas = 0

        for i, bloque in enumerate(bloques):
            try:
                # --- LA MAGIA DEL textContent ---
                # Ahora extraemos el texto desde las entrañas del HTML, 
                # así no importa si la propiedad ya no está visible en pantalla
                nombre = bloque.find_element(By.CSS_SELECTOR, ".poly-component__title").get_attribute("textContent")
                try:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-price__current").get_attribute("textContent")
                except:
                    precio_texto = bloque.find_element(By.CSS_SELECTOR, ".poly-component__price").get_attribute("textContent")

                # Filtro anti-blancos (por si acaso detecta un banner publicitario real)
                if not nombre or not precio_texto or not nombre.strip() or not precio_texto.strip():
                    continue

                try:
                    caracteristicas = bloque.find_element(By.CSS_SELECTOR, ".poly-component__attributes-list").get_attribute("textContent")

                except:
                    caracteristicas = "Sin datos"

                try:
                    direccion = bloque.find_element(By.CSS_SELECTOR, ".poly-component__location").get_attribute("textContent")

                except:
                    direccion = "Dirección no especificada"

                datos_finales.append({
                    "identificador": nombre.strip(),
                    "ciudad" : CIUDAD_ACTUAL,
                    "valor": precio_texto,
                    "caracteristicas": caracteristicas.strip(),
                    "direccion": direccion.strip(),
                    "fecha_captura": time.strftime("%Y-%m-%d %H:%M:%S"),
                    "grupo": NOMBRE_GRUPO
                })
                propiedades_extraidas += 1
                
            except:
                # Si el bloque tiene otro HTML (Proyectos nuevos), lo ignora
                continue
                
        print(f"Se extrajeron {propiedades_extraidas} propiedades reales de {len(bloques)} bloques.")

        # Intenta avanzar a la siguiente página
        try:
            btn_sig = driver.find_element(By.CSS_SELECTOR, ".andes-pagination__button--next a")
            driver.execute_script("arguments[0].click();", btn_sig)
            time.sleep(5)
        except:
            print("No se encontró el botón siguiente o ya es la última página.")
            break

    print(f"\n Extracción terminada: {len(datos_finales)} propiedades obtenidas en total.")

except Exception as e:
    print(f"Error en Selenium: {e}")

finally:
    # Cierra el navegador
    if driver is not None:
        try:
            driver.quit()
        except:
            pass

# --- PASO 3: GUARDAR EN MONGODB ---
try:
    if datos_finales:
        client = MongoClient("database", 27017, serverSelectionTimeoutMS=5000)
        db = client["ArriendosPrueba_IVRegión"]
        coleccion = db["G2_Arriendos"] 

        for d in datos_finales:
            # Limpia el valor antes de convertirlo (Quitamos $, puntos y saltos de línea)
            v_limpio = str(d["valor"]).replace("$", "").replace(".", "").replace(",", "").replace("\n", "").strip()
            d["valor"] = float(v_limpio) if v_limpio.isdigit() else 0.0

        coleccion.insert_many(datos_finales)
        print("Datos cargados en MongoDB correctamente.")
    else:
        print("No hay datos válidos para guardar.")

except Exception as e:
    print(f"Error en MongoDB: {e}")

Limpieza completada. Pantalla virtual configurada.
Navegador iniciado correctamente.

--- Procesando Página 1 ---
Se extrajeron 48 propiedades reales de 48 bloques.

--- Procesando Página 2 ---
Se extrajeron 48 propiedades reales de 48 bloques.

--- Procesando Página 3 ---
Se extrajeron 48 propiedades reales de 48 bloques.

 Extracción terminada: 144 propiedades obtenidas en total.
Datos cargados en MongoDB correctamente.


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, when

# 1. Iniciar sesión (Asegúrate de que la URI sea la correcta)
spark = SparkSession.builder \
    .appName("TablaMaestraInmobiliaria") \
    .config("spark.mongodb.input.uri", "mongodb://database:27017/ArriendosPrueba_IVRegión.G2_Arriendos") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:3.0.1") \
    .getOrCreate()

# 2. Leer datos de MongoDB
df_raw = spark.read.format("mongo").load()

# 3. Estructurar y Limpiar la Tabla
# Aquí "extraemos" los números de los textos de metros cuadrados usando expresiones regulares (RegEx)
df_estructurado = df_raw.select(
    col("identificador").alias("Propiedad"),
    col("ciudad").alias("Ubicacion"),
    col("valor_numerico").alias("Precio_CLP"),
    # Extraemos solo el número de la columna características (ej: "50 m²" -> 50)
    regexp_extract(col("caracteristicas"), r'(\d+)\s*m²', 1).cast("float").alias("Metros_Cuadrados"),
    # Amenidades (ya vienen como True/False del scraping anterior)
    col("piscina"),
    col("quincho"),
    col("lavanderia"),
    col("gimnasio"),
    col("fecha_captura")
).filter(col("Precio_CLP") > 0) # Filtramos basura

# 4. Mostrar la tabla final ordenada por Ciudad y Precio
print("📊 TABLA MAESTRA ESTRUCTURADA:")
df_estructurado.orderBy("Ubicacion", "Precio_CLP").show(20, truncate=False)

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column or function parameter with name `valor_numerico` cannot be resolved. Did you mean one of the following? [`valor`, `_id`, `grupo`, `ciudad`, `direccion`].;
'Project [identificador#6 AS Propiedad#16, ciudad#2 AS Ubicacion#17, 'valor_numerico AS Precio_CLP#18, cast(regexp_extract(caracteristicas#1, (\d+)\s*m², 1) as float) AS Metros_Cuadrados#19, 'piscina, 'quincho, 'lavanderia, 'gimnasio, fecha_captura#4]
+- Relation [_id#0,caracteristicas#1,ciudad#2,direccion#3,fecha_captura#4,grupo#5,identificador#6,valor#7] MongoRelation(MongoRDD[0] at RDD at MongoRDD.scala:51,Some(StructType(StructField(_id,StructType(StructField(oid,StringType,true)),true),StructField(caracteristicas,StringType,true),StructField(ciudad,StringType,true),StructField(direccion,StringType,true),StructField(fecha_captura,StringType,true),StructField(grupo,StringType,true),StructField(identificador,StringType,true),StructField(valor,DoubleType,true))))
